In [1]:
#@title Install Packages and Setup Environment { display-mode: "form" }

from google.colab import files
import zipfile
import io
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

# potential solution
!pip install protobuf==3.20 &> /dev/null
!pip install tensorflow==2.8 &> /dev/null
!apt install --allow-change-held-packages libcudnn8=8.1.0.77-1+cuda11.2 &> /dev/null


!pip -q install streamlit &> /dev/null
!pip -q install pyngrok &> /dev/null
from pyngrok import ngrok

#not necessary for now
#!pip install tensorflowjs
#import tensorflowjs as tfjs

import json
from urllib.request import urlretrieve

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from IPython.display import Image
import os
import gdown

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

def capture_image(filename):
  try:
    take_photo(filename)
    print('Saved to {}'.format(filename))

    # Show the image which was just taken.
    display(Image(filename))
  except Exception as err:
    # Errors will be thrown if the user does not have a webcam or if they do not
    # grant the page permission to access it.
    print(str(err))

import numpy as np
import tensorflow as tf
import cv2
from google.colab.patches import cv2_imshow

# Ensure comptability with different TF versions
version_fn = getattr(tf.keras, "version", None)
if version_fn and version_fn().startswith("3."):
  import tf_keras as keras
else:
  keras = tf.keras

import warnings
warnings.filterwarnings('ignore')


In [2]:
#@title Upload and extract your model into Colab

# upload model zip file
uploaded = files.upload()

# extract model
try:
  file_name = list(uploaded.keys())[-1]
  print("Extracting model...")
  with zipfile.ZipFile(file_name, 'r') as zip_ref:
      zip_ref.extractall('')

  class_names = np.genfromtxt("labels.txt", dtype="str", delimiter='\n')
  for i in range(len(class_names)):
    class_names[i] = ' '.join(class_names[i].split(' ')[1::])

  predictions = [0.0] * len(class_names)

  model = keras.models.load_model('keras_model.h5', compile=False)
  print("Success! Model Extracted!")
except (IndexError, NameError):
  print("Oops! Cannot find file to unzip. Please try uploading a zip file for your model")

Saving converted_keras.zip to converted_keras.zip
Extracting model...
Success! Model Extracted!


In [3]:
print(class_names)

['GRAY LEAF SPOT' 'LEAF BLIGHT MAIZE DISEASE' 'MAIZE STREAK VIRUS'
 'RUST DISEASE']


In [4]:
from google.colab import userdata
!pip install langchain-openai
import langchain
from langchain_openai import ChatOpenAI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.0 MB/s eta 0:00:00


In [5]:
from langchain_openai import OpenAIEmbeddings

In [6]:
 ok_model=ChatOpenAI(
    api_key=userdata.get('Franc'),
    base_url='https://open.bigmodel.cn/api/paas/v4',
    model='glm-4',
    max_completion_tokens=100
  )


In [7]:
embeddings= OpenAIEmbeddings(
    api_key=userdata.get("Franc"),
    base_url="https://open.bigmodel.cn/api/paas/v4",
    model="embedding-3",
)

In [8]:
prompt2= """Yuu are a helpfull assistant for farmers,
 you will be receiving a name of the maize disease and use the given context to answer how does the disease affect
 and the ways to avoid it from maize.
 ***Disease***
  {Disease}
 ***Answer***

 When you recieve the disease you will be answering like this;
 Disease_name:
 Effects:
 Solution:

 """

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain

In [10]:
prompts="""Answer the following questions as best you can. You have access to the following tools:
{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: Analyze the image of a maize leaf showing signs of disease. Identify the specific disease affecting the maize plant, and provide a brief description of its symptoms. Then, recommend effective treatment or prevention methods suitable for smallholder farmers. Respond clearly and concisely.
{agent_scratchpad}
"""

In [11]:
from langchain.tools import tool

In [12]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.4 MB/s eta 0:00:00


In [13]:
# "@tool"
# def Find_disease(image_path):
#   """Analyzes an image of a maize leaf to identify the disease and provide information."""

#   # Load and preprocess the image
#   img = cv2.imread(image_path)
#   img = cv2.resize(img, (224, 224)) # Resize to the input shape of the model
#   img = np.expand_dims(img, axis=0) # Add a batch dimension
#   img = img / 255.0 # Normalize the image if the model requires it

#   # Perform inference
#   predictions = model.predict(img)
#   score = tf.nn.softmax(predictions[0])

#   # Get the predicted class name and confidence
#   predicted_class_index = np.argmax(score)
#   predicted_class_name = class_names[predicted_class_index]
#   confidence = np.max(score)

#   print(f"This image most likely belongs to {predicted_class_name} with a {100 * confidence:.2f}% confidence.")

#   # Use the Search_for_disease_details tool to get more information
#   disease_info = Search_for_disease_details(predicted_class_name)

#   return disease_info, predicted_class_name

In [14]:
# "@tool"
# def Search_for_disease_details(predicted_class_name):
#   """This function will be getting the output from the function of Find_disease as a Maize disease and answer as the prompt says"""
#   prompt_template= PromptTemplate.from_template(template=prompt2)
#   chain= LLMChain(llm=my_model, prompt= prompt_template)
#   resp= chain.invoke(input={'Disease':predicted_class_name})
#   return resp['text']


In [15]:
# prot = PromptTemplate.from_template(prompts)

In [16]:
# tools=[Find_disease, Search_for_disease_details]

In [17]:
# from langchain.agents import create_react_agent, AgentExecutor

In [18]:
# agent = create_react_agent(ok_model,tools=tools,prompt=prot)

In [19]:
# executor= AgentExecutor(agent=agent, tools=tools, verbose=True)

In [20]:
# def process_image_for_gradio(image_path):
#   response= executor.invoke({"input":image_path})
#   return response
#   # disease_info = Find_disease.run(image_path)
#   # return disease_info

In [21]:
import gradio as gr

In [22]:
# def ui():
#     chat_ui = gr.Interface(
#         fn=process_image_for_gradio,
#         inputs=gr.Image(label="Upload Image", type="filepath"),
#         outputs=gr.TextArea(label="Disease Information"),
#         description="GOOD MAIZE AI",
#         title="GOOD MAIZE AI"
#     )
#     chat_ui.launch(debug= True)

# if __name__ == "__main__":
#     ui()


In [24]:
# 🔌 Imports
import cv2
import numpy as np
import tensorflow as tf
import gradio as gr
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import create_react_agent, AgentExecutor


@tool
def Search_for_disease_details(predicted_class_name):
    """Returns detailed information about the maize disease."""
    prompt_template = PromptTemplate.from_template(template=prompt2)
    chain = LLMChain(llm=ok_model, prompt=prompt_template)
    resp = chain.invoke(input={'Disease': predicted_class_name})
    return resp['text']


@tool
def Find_disease(image_path):
    """Analyzes an image of a maize leaf to identify the disease and provide information."""

    # img = cv2.imread(image_path)
    # if img is None:
    #      print("⚠️ ERROR: Unable to load image. Path was:", image_path)
    #      raise ValueError(f"Could not read image from path: {image_path}")
    img= cv2.imread("/content/maiz streak virus.jpg")
    img = cv2.resize(img, (224, 224))
    img = np.expand_dims(img, axis=0)
    img = img.astype(np.float32) / 255.0

    predictions = model.predict(img)
    score = tf.nn.softmax(predictions[0])

    predicted_class_index = np.argmax(score)
    predicted_class_name = class_names[predicted_class_index]
    confidence = float(np.max(score))

    print(f"This image most likely belongs to '{predicted_class_name}' with {confidence * 100:.2f}% confidence.")

    disease_info = Search_for_disease_details(predicted_class_name)
    return disease_info, predicted_class_name


tools = [Find_disease, Search_for_disease_details]
prot = PromptTemplate.from_template(prompts)
agent = create_react_agent(ok_model, tools=tools, prompt=prot)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

def process_image_for_gradio(image_path):
    response = executor.invoke({"input": image_path})
    return response

def ui():
    chat_ui = gr.Interface(
        fn=process_image_for_gradio,
        inputs=gr.Image(label="Upload Image", type="filepath"),
        outputs=gr.TextArea(label="Disease Information"),
        description="GOOD MAIZE AI",
        title="GOOD MAIZE AI"
    )
    chat_ui.launch(debug=True)

# 🚀 Main Entry Point
if __name__ == "__main__":
    ui()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b766bc47db20aa2a72.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)




> Entering new AgentExecutor chain...
Thought: I need to analyze the image of the maize leaf to identify the disease and provide a brief description of its symptoms.
Action: Find_disease
Action Input: image_path (path to the image file)
1/1 [==============================] - 2s 2s/step
This image most likely belongs to 'RUST DISEASE' with 31.10% confidence.


/tmp/ipython-input-884194161.py:42: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  disease_info = Search_for_disease_details(predicted_class_name)
/tmp/ipython-input-884194161.py:15: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=ok_model, prompt=prompt_template)


("Disease_name: Rust Disease\n\nEffects:\nRust disease in maize is caused by several fungi, with the most common being Puccinia sorghi. The disease affects maize by producing reddish-orange to brown pustules on the leaves, husks, and sometimes the stalks. These pustules contain spores that can spread the disease through the wind. The effects of rust disease include:\n\n1. Reduced photosynthesis: The pustules block sunlight from reaching the leaf surface, reducing the plant's ability to photosynthesize and produce energy.\n2. Premature leaf death: Affected leaves may wither and die before the end of the growing season, which can lead to a decrease in yield.\n3. Stunted growth: The overall growth of the plant can be stunted, affecting the size and number of ears produced.\n4. Reduced grain quality: The disease can also affect the quality of the grain, leading to lower market value.\n\nSolution:\nTo avoid rust disease in maize, farmers can implement several strategies:\n\n1. Crop rotation